# IAGA download to K-index plot

This notebook shows the K index calculation steps. It downloads real MAGIE IAGA-2002 files, reads them as a MagPy `DataStream`, computes FMI K-index values, and plots them with `magie.plot_k`.

The example intentionally uses a multi-day archive window because `magpy.core.activity.K_fmi` needs surrounding data to compute K values reliably.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from magpy.core import activity as act

from magie import download_iaga2002, live_k, plot_k
from magie.utils import get_site_metadata

## Configuration

The download range is wider than the final K plot range. That is expected: the K calculation needs context around the plotted interval.

In [ ]:
SITE_CODE = "dun"
START = "2026-01-16T00:00:00"
END = "2026-01-22T23:59:59"

OUTPUT_DIR = Path("/tmp/magie_plot_k_notebook")
DOWNLOAD_DIR = OUTPUT_DIR / "iaga"
FIGURE_PATH = OUTPUT_DIR / "dun_20260116_20260122_k.png"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR

In [ ]:
def stream_count(stream):
    return len(stream["time"])


def time_span(stream):
    times = stream["time"]
    if len(times) == 0:
        return "empty"
    return f"{times[0]} to {times[-1]}"


def selected_time_span(times, mask):
    times = np.asarray(times)
    selected = times[mask]
    if len(selected) == 0:
        return "empty"
    return f"{selected[0]} to {selected[-1]}"

## Download and read IAGA files

`download_iaga2002` downloads each UTC archive day, reads the files with MagPy, joins the daily streams, and returns the joined `DataStream`. `trim=False` keeps the full daily files so the K calculation has enough context.

In [ ]:
data = download_iaga2002(
    START,
    END,
    SITE_CODE,
    save=True,
    save_dir=DOWNLOAD_DIR,
    n_jobs=3,
    trim=False,
    request_timeout=10,
)
data = data.filter()

print(f"Downloaded stream samples: {stream_count(data)}")
print(f"Downloaded stream span: {time_span(data)}")

Inspect a few raw `DataStream` columns directly. This avoids converting the stream to a DataFrame and makes it clear what MagPy is using.

In [ ]:
sample = slice(0, 5)

print("datetime:", data["time"][sample])
print("x:", data["x"][sample])
print("y:", data["y"][sample])
print("z:", data["z"][sample])

## Compute K values

The station metadata supplies the FMI K9 threshold and longitude. Values above 9 are clipped because the plot categories only run from K0 to K9.

In [ ]:
metadata = get_site_metadata(SITE_CODE)

k_data = act.K_fmi(
    data,
    K9_limit=metadata["k9_threshold"],
    longitude=metadata["geodetic_longitude"],
    step_size=60,
)
k_data["var1"] = np.where(k_data["var1"] > 9, 9, k_data["var1"])

if len(k_data["time"]) == 0:
    raise RuntimeError("No K-index values were produced.")

In [ ]:
valid_k = np.asarray(k_data["var1"]) != -1

print(f"K-index samples: {len(k_data['time'])}")
print(f"K-index span: {time_span(k_data)}")
print(f"K-index non--1 samples: {valid_k.sum()}")
print(f"K-index non--1 span: {selected_time_span(k_data['time'], valid_k)}")

print("K times:", k_data["time"])
print("K values:", k_data["var1"])

## Plot the archive K values

This is the non-live plotting path. For wider ranges, `plot_k` keeps the six-hour grid cadence but suppresses the crowded six-hour text labels, leaving the daily date labels.

In [ ]:
fig, ax = plot_k(k_data, colorbar=False)
fig.savefig(FIGURE_PATH, dpi=96, bbox_inches="tight")

xmin, xmax = [mdates.num2date(value).replace(tzinfo=None) for value in ax.get_xlim()]
print(f"Plot x-axis span: {xmin} to {xmax}")
print(f"Saved figure: {FIGURE_PATH}")

fig

## Compare with the live K path

`live_k` now uses the same IAGA downloader when `path_prefix` is HTTPS and `file_format='iaga2002'`. It also tags the returned stream with the intended live plot window, so sparse live stations still show the full live dashboard range.

In [ ]:
LIVE_TIME = pd.Timestamp("2026-10-07T11:50:04")

live_data = live_k(
    LIVE_TIME,
    SITE_CODE,
    path_prefix="https://data.magie.ie/",
    file_format="iaga2002",
)

fig, ax = plot_k(live_data, colorbar=False)
xmin, xmax = [mdates.num2date(value).replace(tzinfo=None) for value in ax.get_xlim()]

print(f"Live K samples: {len(live_data['time'])}")
print(f"Live K span: {time_span(live_data)}")
print("Live plot metadata:", live_data.header.get("plot_k_xlim_start"), live_data.header.get("plot_k_xlim_end"))
print(f"Live plot x-axis span: {xmin} to {xmax}")

fig